# Fake Job Postings: EDA and Classic ML Model Comparison - Colab Version

This Colab-friendly notebook compares the cleaned feature-engineered dataset (`fake_jobs_cleaned.csv`) with the original Kaggle-style dataset (`fake_job_postings.csv`).

Put both CSV files in your shared Google Drive folder. The setup cell mounts Drive and searches for the files automatically. If Colab cannot find the shared folder, add the shared folder as a shortcut to your Google Drive and rerun the setup cell.

This version displays charts and tables inside the notebook but does not save output files or create a local `results/` folder.

Goals:
- Understand class imbalance and missingness.
- Compare text, categorical, and numeric feature patterns across real and fake postings.
- Run stratified cross-validation on several classic machine learning models.
- Use the results to pick a project direction for an imbalanced fake job posting classifier.

In [ ]:
# Core setup - Colab shared Google Drive version with upload fallback
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    make_scorer,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

IN_COLAB = False
try:
    from google.colab import drive, files
    IN_COLAB = True
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive')
except ImportError:
    print('Not running in Colab. Searching from the current working directory instead.')
    DRIVE_ROOT = Path('.')

REQUIRED_FILES = {
    'cleaned': 'fake_jobs_cleaned.csv',
    'original': 'fake_job_postings.csv',
}

# Optional: if you know the folder later, put it here to avoid searching Drive.
# Example: DATA_DIR_OVERRIDE = Path('/content/drive/MyDrive/ML1_Project')
DATA_DIR_OVERRIDE = None


def find_dataset_files(required_files, root, override=None):
    if override is not None:
        paths = {label: override / filename for label, filename in required_files.items()}
        missing = [str(path) for path in paths.values() if not path.exists()]
        if missing:
            raise FileNotFoundError('Could not find these files in DATA_DIR_OVERRIDE:\n' + '\n'.join(missing))
        return paths

    search_roots = [
        Path('/content'),
        Path('/content/drive/MyDrive'),
        Path('/content/drive/Shareddrives'),
        root,
    ]

    found = {}
    for label, filename in required_files.items():
        matches = []
        seen_roots = set()
        for search_root in search_roots:
            if search_root.exists() and search_root not in seen_roots:
                seen_roots.add(search_root)
                matches.extend(search_root.rglob(filename))
            if matches:
                break
        if matches:
            found[label] = matches[0]
    return found


def upload_missing_files(required_files, already_found):
    missing = {label: filename for label, filename in required_files.items() if label not in already_found}
    if not missing:
        return already_found

    if not IN_COLAB:
        missing_files = ', '.join(missing.values())
        raise FileNotFoundError('Could not find: ' + missing_files)

    print('Could not find these files through the mounted Drive path:')
    for filename in missing.values():
        print(f'- {filename}')
    print('\nColab does not always expose files from Shared with me folders.')
    print('Please upload the missing CSV files when prompted, or add the shared folder as a shortcut to My Drive and rerun this cell.')

    uploaded = files.upload()
    for label, filename in missing.items():
        uploaded_path = Path('/content') / filename
        local_path = Path(filename)
        if uploaded_path.exists():
            already_found[label] = uploaded_path
        elif local_path.exists():
            already_found[label] = local_path

    still_missing = [filename for label, filename in required_files.items() if label not in already_found]
    if still_missing:
        raise FileNotFoundError('Still missing after upload: ' + ', '.join(still_missing))
    return already_found


DATASETS = find_dataset_files(REQUIRED_FILES, DRIVE_ROOT, DATA_DIR_OVERRIDE)
DATASETS = upload_missing_files(REQUIRED_FILES, DATASETS)

TARGET = 'fraudulent'
RANDOM_STATE = 42
N_SPLITS = 5

print('Datasets found:')
for label, dataset_path in DATASETS.items():
    print(f'- {label}: {dataset_path}')

In [ ]:
# Helper functions
TEXT_COLUMNS = ['title', 'company_profile', 'description', 'requirements', 'benefits']
BASE_CATEGORICAL_COLUMNS = [
    'location', 'department', 'employment_type', 'required_experience',
    'required_education', 'industry', 'function'
]
ID_COLUMNS = ['job_id']


def slugify(value):
    return re.sub(r'[^a-zA-Z0-9_]+', '_', str(value)).strip('_').lower()


def savefig(name):
    # Colab version: keep figures embedded in the notebook only.
    plt.tight_layout()
    return None


def display_and_save_table(df, name, index=True):
    # Colab version: display tables only; do not write CSV outputs.
    display(df)


def load_dataset(path):
    df = pd.read_csv(path)
    if TARGET not in df.columns:
        raise ValueError(f'Missing target column: {TARGET}')
    return df


def prepare_features(df):
    X = df.drop(columns=[TARGET]).copy()
    available_text = [c for c in TEXT_COLUMNS if c in X.columns]
    for col in available_text:
        X[col] = X[col].fillna('').astype(str)
    X['text_all'] = X[available_text].agg(' '.join, axis=1) if available_text else ''

    # Keep original text columns out of the structured transformer because text_all contains them.
    drop_cols = [c for c in ID_COLUMNS + available_text if c in X.columns]
    X_model = X.drop(columns=drop_cols, errors='ignore')

    categorical_cols = [c for c in BASE_CATEGORICAL_COLUMNS if c in X_model.columns]
    numeric_cols = [c for c in X_model.columns if c not in categorical_cols + ['text_all']]

    # Some engineered columns may be read as object if they contain unusual values; keep only true numeric columns.
    numeric_cols = [c for c in numeric_cols if pd.api.types.is_numeric_dtype(X_model[c])]
    passthrough_cols = ['text_all'] + categorical_cols + numeric_cols
    X_model = X_model[passthrough_cols]
    return X_model, df[TARGET].astype(int), categorical_cols, numeric_cols


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True)


def make_preprocessor(categorical_cols, numeric_cols, use_text=True, use_metadata=True):
    transformers = []
    if use_text:
        transformers.append((
            'text',
            TfidfVectorizer(
                lowercase=True,
                stop_words='english',
                max_features=6000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
            'text_all'
        ))
    if use_metadata and categorical_cols:
        transformers.append((
            'categorical',
            Pipeline([
                ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', make_one_hot_encoder()),
            ]),
            categorical_cols
        ))
    if use_metadata and numeric_cols:
        transformers.append((
            'numeric',
            Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', MaxAbsScaler()),
            ]),
            numeric_cols
        ))
    return ColumnTransformer(transformers=transformers, remainder='drop')


def make_models(categorical_cols, numeric_cols):
    text_metadata = make_preprocessor(categorical_cols, numeric_cols, use_text=True, use_metadata=True)
    text_only = make_preprocessor(categorical_cols, numeric_cols, use_text=True, use_metadata=False)
    metadata_only = make_preprocessor(categorical_cols, numeric_cols, use_text=False, use_metadata=True)

    return {
        'Dummy majority baseline': Pipeline([
            ('features', text_metadata),
            ('model', DummyClassifier(strategy='most_frequent')),
        ]),
        'Logistic Regression balanced': Pipeline([
            ('features', text_metadata),
            ('model', LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear', random_state=RANDOM_STATE)),
        ]),
        'Linear SVM balanced': Pipeline([
            ('features', text_metadata),
            ('model', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
        ]),
        'Complement Naive Bayes text only': Pipeline([
            ('features', text_only),
            ('model', ComplementNB(alpha=0.5)),
        ]),
        'Random Forest metadata only': Pipeline([
            ('features', metadata_only),
            ('model', RandomForestClassifier(
                n_estimators=250,
                class_weight='balanced_subsample',
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )),
        ]),
    }


def positive_precision(y_true, y_pred):
    return precision_score(y_true, y_pred, pos_label=1, zero_division=0)


def positive_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=1, zero_division=0)


def positive_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, pos_label=1, zero_division=0)

SCORING = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'precision_fake': make_scorer(positive_precision),
    'recall_fake': make_scorer(positive_recall),
    'f1_fake': make_scorer(positive_f1),
}

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# Load datasets
raw_datasets = {name: load_dataset(path) for name, path in DATASETS.items()}

summary_rows = []
for name, df in raw_datasets.items():
    counts = df[TARGET].value_counts().sort_index()
    summary_rows.append({
        'dataset': name,
        'rows': len(df),
        'columns': df.shape[1],
        'real_count': int(counts.get(0, 0)),
        'fake_count': int(counts.get(1, 0)),
        'fake_rate': counts.get(1, 0) / len(df),
    })

summary = pd.DataFrame(summary_rows)
display_and_save_table(summary, 'dataset_summary', index=False)

## Exploratory Data Analysis

The EDA below is repeated for the cleaned and original datasets. Because fake postings are the minority class, most charts separate real and fake postings rather than only showing overall averages.

In [ ]:
# EDA: class balance and missingness
for name, df in raw_datasets.items():
    display(Markdown(f'### {name.title()} Dataset'))
    print(f'Shape: {df.shape}')
    print('Columns:', ', '.join(df.columns))

    class_counts = df[TARGET].map({0: 'Real', 1: 'Fake'}).value_counts().reindex(['Real', 'Fake'])
    class_rate = (class_counts / class_counts.sum()).rename('rate')
    class_table = pd.DataFrame({'count': class_counts, 'rate': class_rate})
    display_and_save_table(class_table, f'{name}_class_balance')

    plt.figure(figsize=(5, 4))
    ax = sns.barplot(x=class_counts.index, y=class_counts.values)
    ax.set_title(f'{name.title()} Class Balance')
    ax.set_ylabel('Number of postings')
    ax.set_xlabel('Label')
    for i, val in enumerate(class_counts.values):
        ax.text(i, val, f'{val:,}', ha='center', va='bottom')
    savefig(f'{name}_class_balance')
    plt.show()

    missing = df.isna().mean().sort_values(ascending=False).head(15).rename('missing_rate').to_frame()
    display_and_save_table(missing, f'{name}_missing_rates_top15')

    plt.figure(figsize=(8, 5))
    sns.barplot(data=missing.reset_index(), y='index', x='missing_rate', color='#6aaed6')
    plt.title(f'{name.title()} Top Missing Rates')
    plt.xlabel('Missing rate')
    plt.ylabel('Column')
    savefig(f'{name}_missing_rates_top15')
    plt.show()

In [ ]:
# EDA: text length patterns
for name, df in raw_datasets.items():
    display(Markdown(f'### Text Length Patterns: {name.title()}'))
    text_cols = [c for c in TEXT_COLUMNS if c in df.columns]
    length_frame = pd.DataFrame({
        col: df[col].fillna('').astype(str).str.len()
        for col in text_cols
    })
    length_frame[TARGET] = df[TARGET].map({0: 'Real', 1: 'Fake'})

    melted = length_frame.melt(id_vars=TARGET, var_name='field', value_name='characters')
    length_summary = melted.groupby([TARGET, 'field'])['characters'].agg(['mean', 'median', 'max']).round(1)
    display_and_save_table(length_summary, f'{name}_text_length_summary')

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=melted, x='field', y='characters', hue=TARGET, showfliers=False)
    plt.title(f'{name.title()} Text Length by Label')
    plt.xlabel('Text field')
    plt.ylabel('Characters, outliers hidden')
    plt.xticks(rotation=25, ha='right')
    savefig(f'{name}_text_length_by_label')
    plt.show()

In [ ]:
# EDA: categorical fields with the highest fake rates
for name, df in raw_datasets.items():
    display(Markdown(f'### Categorical Fake Rates: {name.title()}'))
    cat_cols = [c for c in BASE_CATEGORICAL_COLUMNS if c in df.columns]
    frames = []
    for col in cat_cols:
        temp = df[[col, TARGET]].copy()
        temp[col] = temp[col].fillna('missing').astype(str)
        grouped = temp.groupby(col)[TARGET].agg(['count', 'mean']).rename(columns={'mean': 'fake_rate'})
        grouped = grouped[grouped['count'] >= 30].sort_values('fake_rate', ascending=False).head(8)
        grouped.insert(0, 'column', col)
        grouped.insert(1, 'value', grouped.index)
        frames.append(grouped.reset_index(drop=True))
    cat_rates = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    display_and_save_table(cat_rates, f'{name}_top_categorical_fake_rates', index=False)

    if not cat_rates.empty:
        plot_data = cat_rates.head(20).copy()
        plot_data['label'] = plot_data['column'] + ': ' + plot_data['value'].str.slice(0, 35)
        plt.figure(figsize=(9, 7))
        sns.barplot(data=plot_data, y='label', x='fake_rate', hue='column', dodge=False)
        plt.title(f'{name.title()} Highest Fake Rates Among Common Categories')
        plt.xlabel('Fake rate')
        plt.ylabel('Category')
        plt.legend(loc='lower right')
        savefig(f'{name}_top_categorical_fake_rates')
        plt.show()

## Cross-Validated Model Comparison

The models below are intentionally classic and explainable enough for an introductory machine learning project:

- Dummy majority baseline: shows why accuracy is not enough.
- Logistic Regression with class weights: strong baseline for sparse text features.
- Linear SVM with class weights: another strong linear text classifier.
- Complement Naive Bayes: simple text-only baseline often used for imbalanced text classification.
- Random Forest on metadata only: tests how far non-text structured fields can go.

The main ranking metric is `average_precision` because it focuses on finding the rare fake class across decision thresholds.

In [ ]:
# Run cross-validation comparisons
all_results = []
fitted_feature_info = []

for dataset_name, df in raw_datasets.items():
    display(Markdown(f'### Cross-validation: {dataset_name.title()}'))
    X, y, categorical_cols, numeric_cols = prepare_features(df)
    models = make_models(categorical_cols, numeric_cols)
    fitted_feature_info.append({
        'dataset': dataset_name,
        'categorical_features': ', '.join(categorical_cols),
        'numeric_features': ', '.join(numeric_cols),
        'positive_class_rate': y.mean(),
    })
    print('Categorical features:', categorical_cols)
    print('Numeric features:', numeric_cols)

    for model_name, estimator in models.items():
        print(f'Running {dataset_name} | {model_name}...')
        scores = cross_validate(
            estimator,
            X,
            y,
            cv=cv,
            scoring=SCORING,
            n_jobs=-1,
            return_train_score=False,
            error_score='raise',
        )
        row = {'dataset': dataset_name, 'model': model_name}
        for metric_name in SCORING:
            values = scores[f'test_{metric_name}']
            row[f'{metric_name}_mean'] = values.mean()
            row[f'{metric_name}_std'] = values.std()
        all_results.append(row)

cv_results = pd.DataFrame(all_results)
metric_cols = [c for c in cv_results.columns if c.endswith('_mean')]
cv_results_sorted = cv_results.sort_values(['dataset', 'average_precision_mean'], ascending=[True, False])
display_and_save_table(cv_results_sorted.round(4), 'cv_model_comparison', index=False)

feature_info = pd.DataFrame(fitted_feature_info)
display_and_save_table(feature_info, 'model_feature_columns', index=False)

In [ ]:
# Plot CV results
plot_metrics = ['average_precision_mean', 'roc_auc_mean', 'f1_fake_mean', 'recall_fake_mean', 'balanced_accuracy_mean']
plot_data = cv_results.melt(
    id_vars=['dataset', 'model'],
    value_vars=plot_metrics,
    var_name='metric',
    value_name='score'
)
plot_data['metric'] = plot_data['metric'].str.replace('_mean', '', regex=False).str.replace('_', ' ').str.title()

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_data, x='score', y='model', hue='metric')
plt.title('Cross-Validated Model Scores')
plt.xlabel('Mean CV score')
plt.ylabel('Model')
plt.xlim(0, 1)
plt.legend(loc='lower right')
savefig('cv_model_scores_all_metrics')
plt.show()

plt.figure(figsize=(9, 5))
sns.barplot(data=cv_results, x='average_precision_mean', y='model', hue='dataset')
plt.title('Average Precision by Dataset and Model')
plt.xlabel('Mean average precision across folds')
plt.ylabel('Model')
plt.xlim(0, 1)
savefig('average_precision_by_dataset_model')
plt.show()

In [ ]:
# Confusion matrices and classification reports for the best model per dataset
best_rows = cv_results.sort_values('average_precision_mean', ascending=False).groupby('dataset').head(1)
reports = []

for _, best in best_rows.iterrows():
    dataset_name = best['dataset']
    model_name = best['model']
    df = raw_datasets[dataset_name]
    X, y, categorical_cols, numeric_cols = prepare_features(df)
    estimator = make_models(categorical_cols, numeric_cols)[model_name]

    display(Markdown(f'### Best Model by Average Precision: {dataset_name.title()}'))
    print(model_name)

    y_pred = cross_val_predict(estimator, X, y, cv=cv, n_jobs=-1, method='predict')
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    report_dict = classification_report(y, y_pred, labels=[0, 1], target_names=['Real', 'Fake'], output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).T
    report_df.insert(0, 'dataset', dataset_name)
    report_df.insert(1, 'model', model_name)
    reports.append(report_df.reset_index(names='class'))

    display(report_df.round(4))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'Fake'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'{dataset_name.title()} Best Model Confusion Matrix\n{model_name}')
    savefig(f'{dataset_name}_best_model_confusion_matrix')
    plt.show()

classification_reports = pd.concat(reports, ignore_index=True)
display_and_save_table(classification_reports.round(4), 'best_model_classification_reports', index=False)

In [ ]:
# Simple project-direction summary generated from results
top_by_dataset = cv_results.sort_values('average_precision_mean', ascending=False).groupby('dataset').head(1)
comparison = top_by_dataset[['dataset', 'model', 'average_precision_mean', 'roc_auc_mean', 'f1_fake_mean', 'recall_fake_mean', 'precision_fake_mean']].round(4)
display_and_save_table(comparison, 'top_model_by_dataset', index=False)

cleaned_top = comparison[comparison['dataset'] == 'cleaned'].iloc[0]
original_top = comparison[comparison['dataset'] == 'original'].iloc[0]

print('Suggested path forward:')
print(f"1. Use average precision as the main comparison metric because the fake class is only about {summary.loc[0, 'fake_rate']:.1%} of the data.")
print(f"2. Treat {cleaned_top['model']} on the cleaned dataset as the current leading baseline if interpretability and performance are both acceptable.")
print(f"3. Compare cleaned vs original results directly: cleaned AP={cleaned_top['average_precision_mean']}, original AP={original_top['average_precision_mean']}.")
print('4. For the class project, a strong next step is error analysis: inspect false positives and false negatives, then explain which words/metadata patterns drive the model.')
print('5. Avoid presenting accuracy as the headline metric; the dummy baseline can look high simply because most postings are real.')

## Project Framing: Beyond Accuracy

**Core research question:** How does class imbalance affect model evaluation and model selection when detecting fraudulent job postings?

Because fake postings are only about 4.84% of the dataset, a model can achieve high accuracy while missing most fake jobs. This project therefore treats accuracy as a secondary metric and focuses on minority-class performance.

**Research questions:**

1. How imbalanced is the dataset, and why does that matter?
2. Is accuracy misleading for fake job detection?
3. Which classic machine learning model performs best on the minority fake class?
4. Does the cleaned and feature-engineered dataset improve model performance compared with the original dataset?
5. Which evaluation metric gives the most useful view of model performance?
6. What tradeoff exists between catching fake jobs and falsely flagging real jobs?
7. Do class-weighted models perform better than unweighted models on minority-class metrics?
8. Which words or metadata features appear most predictive of fraudulent postings?
9. What kinds of postings are most likely to become false positives or false negatives?

**Hypotheses:**

- H1: Accuracy will overestimate model usefulness because of severe class imbalance.
- H2: Class-weighted linear models will outperform simple baselines on minority-class metrics.
- H3: Text-based models will outperform metadata-only models because fraudulent language patterns are embedded in job descriptions, requirements, and company profiles.
- H4: The cleaned dataset will improve performance slightly, but not dramatically, because TF-IDF text features already capture much of the useful signal.

## Weighted vs Unweighted Linear Models

Class weighting is one of the simplest ways to address class imbalance in classic machine learning. The next experiment compares unweighted and class-weighted versions of Logistic Regression and Linear SVM. If the imbalance hypothesis is correct, class-weighted models should improve recall and F1 for the fake class, even if accuracy changes only slightly.

In [ ]:
# Compare weighted vs unweighted linear models
weighted_results = []

for dataset_name, df in raw_datasets.items():
    X, y, categorical_cols, numeric_cols = prepare_features(df)
    preprocessor = make_preprocessor(categorical_cols, numeric_cols, use_text=True, use_metadata=True)
    weighted_models = {
        'Logistic Regression unweighted': Pipeline([
            ('features', clone(preprocessor)),
            ('model', LogisticRegression(max_iter=1000, solver='liblinear', random_state=RANDOM_STATE)),
        ]),
        'Logistic Regression balanced': Pipeline([
            ('features', clone(preprocessor)),
            ('model', LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear', random_state=RANDOM_STATE)),
        ]),
        'Linear SVM unweighted': Pipeline([
            ('features', clone(preprocessor)),
            ('model', LinearSVC(random_state=RANDOM_STATE)),
        ]),
        'Linear SVM balanced': Pipeline([
            ('features', clone(preprocessor)),
            ('model', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
        ]),
    }

    for model_name, estimator in weighted_models.items():
        print(f'Running {dataset_name} | {model_name}...')
        scores = cross_validate(
            estimator,
            X,
            y,
            cv=cv,
            scoring=SCORING,
            n_jobs=-1,
            return_train_score=False,
            error_score='raise',
        )
        row = {'dataset': dataset_name, 'model': model_name}
        for metric_name in SCORING:
            values = scores[f'test_{metric_name}']
            row[f'{metric_name}_mean'] = values.mean()
            row[f'{metric_name}_std'] = values.std()
        weighted_results.append(row)

weighted_results_df = pd.DataFrame(weighted_results).sort_values(['dataset', 'average_precision_mean'], ascending=[True, False])
display_and_save_table(weighted_results_df.round(4), 'weighted_vs_unweighted_linear_models', index=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=weighted_results_df, x='f1_fake_mean', y='model', hue='dataset')
plt.title('Weighted vs Unweighted Linear Models: Fake-Class F1')
plt.xlabel('Mean fake-class F1 across folds')
plt.ylabel('Model')
plt.xlim(0, 1)
savefig('weighted_vs_unweighted_fake_f1')
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=weighted_results_df, x='recall_fake_mean', y='model', hue='dataset')
plt.title('Weighted vs Unweighted Linear Models: Fake-Class Recall')
plt.xlabel('Mean fake-class recall across folds')
plt.ylabel('Model')
plt.xlim(0, 1)
savefig('weighted_vs_unweighted_fake_recall')
plt.show()

## Precision-Recall Curves

For imbalanced classification problems, precision-recall curves are often more informative than accuracy. The curve shows the tradeoff between catching more fake jobs and avoiding false alarms among real postings.

In [ ]:
# Precision-recall curves for the best model on each dataset
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

for _, best in best_rows.iterrows():
    dataset_name = best['dataset']
    model_name = best['model']
    df = raw_datasets[dataset_name]
    X, y, categorical_cols, numeric_cols = prepare_features(df)
    estimator = make_models(categorical_cols, numeric_cols)[model_name]

    if hasattr(estimator[-1], 'decision_function'):
        y_score = cross_val_predict(estimator, X, y, cv=cv, n_jobs=-1, method='decision_function')
    else:
        y_score = cross_val_predict(estimator, X, y, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]

    ap = average_precision_score(y, y_score)
    PrecisionRecallDisplay.from_predictions(y, y_score, pos_label=1)
    plt.title(f'{dataset_name.title()} Precision-Recall Curve\n{model_name} | AP={ap:.3f}')
    savefig(f'{dataset_name}_best_model_precision_recall_curve')
    plt.show()

## Feature Interpretation

Linear models are useful for this project because they allow basic feature interpretation. Positive coefficients push a posting toward the fake class, while negative coefficients push it toward the real class. These features should not be treated as universal proof of fraud, but they help explain what the model learned from this dataset.

In [ ]:
# Interpret top features from the best cleaned linear model
interpret_dataset = 'cleaned'
interpret_model_name = 'Linear SVM balanced'

df = raw_datasets[interpret_dataset]
X, y, categorical_cols, numeric_cols = prepare_features(df)
interpret_model = make_models(categorical_cols, numeric_cols)[interpret_model_name]
interpret_model.fit(X, y)

feature_names = interpret_model.named_steps['features'].get_feature_names_out()
coefs = interpret_model.named_steps['model'].coef_.ravel()
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['feature'] = coef_df['feature'].str.replace('text__', 'text: ', regex=False)
coef_df['feature'] = coef_df['feature'].str.replace('categorical__', 'category: ', regex=False)
coef_df['feature'] = coef_df['feature'].str.replace('numeric__', 'numeric: ', regex=False)

top_fake_features = coef_df.sort_values('coefficient', ascending=False).head(25)
top_real_features = coef_df.sort_values('coefficient', ascending=True).head(25)

display(Markdown('### Features Most Associated With Fake Postings'))
display_and_save_table(top_fake_features.round(4), 'top_features_associated_with_fake', index=False)

display(Markdown('### Features Most Associated With Real Postings'))
display_and_save_table(top_real_features.round(4), 'top_features_associated_with_real', index=False)

plot_features = pd.concat([
    top_fake_features.assign(direction='Fake'),
    top_real_features.assign(direction='Real')
], ignore_index=True)
plot_features['abs_coefficient'] = plot_features['coefficient'].abs()
plot_features['short_feature'] = plot_features['feature'].str.slice(0, 55)

plt.figure(figsize=(10, 9))
sns.barplot(data=plot_features, x='abs_coefficient', y='short_feature', hue='direction')
plt.title('Top Linear SVM Feature Weights')
plt.xlabel('Absolute coefficient value')
plt.ylabel('Feature')
savefig('top_linear_svm_feature_weights')
plt.show()

## Error Analysis

The final step is to inspect examples the model gets wrong. False positives are real jobs incorrectly flagged as fake. False negatives are fake jobs that the model misses. This is where the project can connect model metrics to practical consequences.

In [ ]:
# Error analysis for the best cleaned model
analysis_dataset = 'cleaned'
analysis_model_name = 'Linear SVM balanced'

df = raw_datasets[analysis_dataset].copy()
X, y, categorical_cols, numeric_cols = prepare_features(df)
analysis_model = make_models(categorical_cols, numeric_cols)[analysis_model_name]

y_pred = cross_val_predict(analysis_model, X, y, cv=cv, n_jobs=-1, method='predict')
y_score = cross_val_predict(analysis_model, X, y, cv=cv, n_jobs=-1, method='decision_function')

error_df = df.copy()
error_df['actual_label'] = y.map({0: 'Real', 1: 'Fake'})
error_df['predicted_label'] = pd.Series(y_pred).map({0: 'Real', 1: 'Fake'})
error_df['model_score_for_fake'] = y_score

for col in ['title', 'location', 'employment_type', 'required_experience', 'industry', 'function', 'description']:
    if col not in error_df.columns:
        error_df[col] = ''
error_df['description_snippet'] = error_df['description'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 240)

cols_to_show = [
    'job_id', 'title', 'location', 'employment_type', 'required_experience',
    'industry', 'function', 'actual_label', 'predicted_label',
    'model_score_for_fake', 'description_snippet'
]

false_positives = error_df[(y == 0) & (y_pred == 1)].sort_values('model_score_for_fake', ascending=False)[cols_to_show].head(10)
false_negatives = error_df[(y == 1) & (y_pred == 0)].sort_values('model_score_for_fake', ascending=True)[cols_to_show].head(10)

error_summary = pd.DataFrame({
    'error_type': ['False positives', 'False negatives'],
    'count': [int(((y == 0) & (y_pred == 1)).sum()), int(((y == 1) & (y_pred == 0)).sum())],
})
display_and_save_table(error_summary, 'cleaned_best_model_error_summary', index=False)

display(Markdown('### Highest-Scoring False Positives'))
display_and_save_table(false_positives, 'cleaned_best_model_false_positives', index=False)

display(Markdown('### Lowest-Scoring False Negatives'))
display_and_save_table(false_negatives, 'cleaned_best_model_false_negatives', index=False)

## Expanded Error Analysis: Where the Model Fails

The earlier confusion matrix tells us how many mistakes the model made. This section goes further and asks what kinds of postings are involved in those mistakes.

The analysis focuses on four prediction groups:

- **True positives:** fake postings correctly flagged as fake.
- **False negatives:** fake postings missed by the model.
- **False positives:** real postings incorrectly flagged as fake.
- **True negatives:** real postings correctly ignored.

This helps answer the practical question behind the project: is the model making understandable mistakes, and what tradeoff exists between catching more scams and falsely flagging legitimate jobs?

In [ ]:
# Expanded error analysis: four prediction groups and feature patterns
analysis_dataset = 'cleaned'
analysis_model_name = 'Linear SVM balanced'

df = raw_datasets[analysis_dataset].copy()
X, y, categorical_cols, numeric_cols = prepare_features(df)
analysis_model = make_models(categorical_cols, numeric_cols)[analysis_model_name]

y_pred = cross_val_predict(analysis_model, X, y, cv=cv, n_jobs=-1, method='predict')
y_score = cross_val_predict(analysis_model, X, y, cv=cv, n_jobs=-1, method='decision_function')

error_df = df.copy()
error_df['actual'] = y.values
error_df['predicted'] = y_pred
error_df['actual_label'] = error_df['actual'].map({0: 'Real', 1: 'Fake'})
error_df['predicted_label'] = error_df['predicted'].map({0: 'Real', 1: 'Fake'})
error_df['model_score_for_fake'] = y_score

conditions = [
    (error_df['actual'].eq(1) & error_df['predicted'].eq(1)),
    (error_df['actual'].eq(1) & error_df['predicted'].eq(0)),
    (error_df['actual'].eq(0) & error_df['predicted'].eq(1)),
    (error_df['actual'].eq(0) & error_df['predicted'].eq(0)),
]
choices = ['True positive', 'False negative', 'False positive', 'True negative']
error_df['prediction_group'] = np.select(conditions, choices, default='Other')

group_order = ['True positive', 'False negative', 'False positive', 'True negative']
error_df['prediction_group'] = pd.Categorical(error_df['prediction_group'], categories=group_order, ordered=True)

# Ensure common analysis columns exist, even if a dataset version lacks one.
for col in ['title', 'description', 'requirements', 'company_profile', 'benefits']:
    if col not in error_df.columns:
        error_df[col] = ''
    error_df[f'{col}_chars_for_analysis'] = error_df[col].fillna('').astype(str).str.len()

for col in ['has_company_logo', 'has_questions', 'has_salary_range', 'has_benefits', 'has_company_profile', 'has_department']:
    if col not in error_df.columns:
        if col == 'has_salary_range' and 'salary_range' in error_df.columns:
            error_df[col] = error_df['salary_range'].notna().astype(int)
        elif col == 'has_benefits' and 'benefits' in error_df.columns:
            error_df[col] = error_df['benefits'].notna().astype(int)
        elif col == 'has_company_profile' and 'company_profile' in error_df.columns:
            error_df[col] = error_df['company_profile'].notna().astype(int)
        elif col == 'has_department' and 'department' in error_df.columns:
            error_df[col] = error_df['department'].notna().astype(int)
        else:
            error_df[col] = np.nan

summary_cols = [
    'model_score_for_fake',
    'title_chars_for_analysis',
    'description_chars_for_analysis',
    'requirements_chars_for_analysis',
    'company_profile_chars_for_analysis',
    'benefits_chars_for_analysis',
    'has_company_logo',
    'has_questions',
    'has_salary_range',
    'has_benefits',
    'has_company_profile',
    'has_department',
]

error_group_summary = error_df.groupby('prediction_group', observed=False)[summary_cols].mean().round(3)
error_group_summary.insert(0, 'count', error_df['prediction_group'].value_counts().reindex(group_order).fillna(0).astype(int))

display_and_save_table(error_group_summary, 'cleaned_error_group_feature_summary')

plt.figure(figsize=(8, 4))
sns.countplot(data=error_df, y='prediction_group', order=group_order, color='#6aaed6')
plt.title('Prediction Groups for Cleaned Dataset Best Model')
plt.xlabel('Number of postings')
plt.ylabel('Prediction group')
savefig('cleaned_error_group_counts')
plt.show()

rate_cols = ['has_company_logo', 'has_questions', 'has_salary_range', 'has_benefits', 'has_company_profile', 'has_department']
rate_plot = error_group_summary[rate_cols].reset_index().melt(id_vars='prediction_group', var_name='feature', value_name='rate')
plt.figure(figsize=(11, 6))
sns.barplot(data=rate_plot, x='rate', y='feature', hue='prediction_group')
plt.title('Feature Presence Rates by Prediction Group')
plt.xlabel('Mean rate')
plt.ylabel('Feature')
plt.xlim(0, 1)
savefig('cleaned_error_group_feature_rates')
plt.show()

### Error Groups by Category

The next table looks at common categorical fields within each prediction group. This helps identify whether certain industries, job functions, employment types, or experience levels are more common among mistakes.

In [ ]:
# Top categorical values within each prediction group
category_columns_for_errors = ['industry', 'function', 'employment_type', 'required_experience', 'required_education']
category_rows = []

for group in group_order:
    group_df = error_df[error_df['prediction_group'].eq(group)]
    for col in category_columns_for_errors:
        if col in group_df.columns:
            counts = group_df[col].fillna('missing').astype(str).value_counts().head(5)
            for value, count in counts.items():
                category_rows.append({
                    'prediction_group': group,
                    'column': col,
                    'value': value,
                    'count': int(count),
                    'group_rate': round(count / len(group_df), 4) if len(group_df) else 0,
                })

error_group_top_categories = pd.DataFrame(category_rows)
display_and_save_table(error_group_top_categories, 'cleaned_error_group_top_categories', index=False)

# Compact display for the most project-relevant mistake groups.
for group in ['False positive', 'False negative']:
    display(Markdown(f'#### Top categories: {group}'))
    display(error_group_top_categories[error_group_top_categories['prediction_group'].eq(group)].head(15))

## Example-Based Error Discussion

Tables are helpful, but individual examples make the error analysis easier to explain in a presentation or report. The examples below show the most confident false positives and false negatives, along with a short model-facing interpretation.

- **False positives:** real jobs that looked suspicious to the model.
- **False negatives:** fake jobs that looked legitimate enough to escape the model.

In [ ]:
# Example-based discussion of false positives and false negatives
for col in ['job_id', 'title', 'location', 'employment_type', 'required_experience', 'industry', 'function', 'description']:
    if col not in error_df.columns:
        error_df[col] = ''

error_df['description_snippet'] = (
    error_df['description']
    .fillna('')
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.slice(0, 260)
)

def explain_error(row):
    reasons = []
    desc = str(row.get('description', '')).lower()
    title = str(row.get('title', '')).lower()
    combined = f'{title} {desc}'
    if any(word in combined for word in ['money', 'earn', 'link', 'click', 'home', 'remote']):
        reasons.append('contains terms that may resemble scam or remote-work language')
    if pd.isna(row.get('company_profile')) or str(row.get('company_profile', '')).strip() == '':
        reasons.append('missing company profile')
    if pd.isna(row.get('requirements')) or str(row.get('requirements', '')).strip() == '':
        reasons.append('missing requirements text')
    if len(str(row.get('description', ''))) < 400:
        reasons.append('short description')
    if row.get('has_company_logo', np.nan) == 0:
        reasons.append('no company logo')
    if not reasons:
        reasons.append('likely reflects subtler text or category patterns learned by the model')
    return '; '.join(reasons[:3])

cols_to_show = [
    'job_id', 'title', 'location', 'employment_type', 'required_experience',
    'industry', 'function', 'actual_label', 'predicted_label',
    'model_score_for_fake', 'description_snippet'
]

false_positive_examples = (
    error_df[error_df['prediction_group'].eq('False positive')]
    .sort_values('model_score_for_fake', ascending=False)
    .head(6)
    .copy()
)
false_negative_examples = (
    error_df[error_df['prediction_group'].eq('False negative')]
    .sort_values('model_score_for_fake', ascending=True)
    .head(6)
    .copy()
)

false_positive_examples['possible_reason_model_was_confused'] = false_positive_examples.apply(explain_error, axis=1)
false_negative_examples['possible_reason_model_was_confused'] = false_negative_examples.apply(explain_error, axis=1)

example_discussion = pd.concat([
    false_positive_examples.assign(error_type='False positive'),
    false_negative_examples.assign(error_type='False negative'),
], ignore_index=True)

example_cols = ['error_type'] + cols_to_show + ['possible_reason_model_was_confused']
example_discussion = example_discussion[example_cols]

display_and_save_table(example_discussion, 'cleaned_error_examples_for_discussion', index=False)

for error_type in ['False positive', 'False negative']:
    display(Markdown(f'### {error_type} examples'))
    display(example_discussion[example_discussion['error_type'].eq(error_type)].head(6))

## Threshold Analysis: Catch More Fakes or Reduce False Alarms?

A real fake-job detector would probably produce a review score rather than a final automatic decision. By changing the decision threshold, we can make the model more aggressive or more conservative.

- A lower threshold flags more postings as fake. This usually increases fake recall but creates more false positives.
- A higher threshold flags fewer postings as fake. This usually increases precision but misses more fake postings.

This threshold analysis directly supports the imbalance-focused project angle.

In [ ]:
# Threshold tradeoff analysis for the balanced Linear SVM decision scores
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, balanced_accuracy_score

threshold_rows = []
# Use desired flag rates plus the model's default threshold of 0.
flag_rates = [0.03, 0.04, 0.05, 0.075, 0.10, 0.15]
thresholds = sorted({float(np.quantile(y_score, 1 - rate)) for rate in flag_rates} | {0.0})

for threshold in thresholds:
    threshold_pred = (y_score >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y, threshold_pred, labels=[1], average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, threshold_pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        'threshold': threshold,
        'flagged_count': int(threshold_pred.sum()),
        'flagged_rate': threshold_pred.mean(),
        'true_positives': int(tp),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_negatives': int(tn),
        'accuracy': accuracy_score(y, threshold_pred),
        'balanced_accuracy': balanced_accuracy_score(y, threshold_pred),
        'fake_precision': precision,
        'fake_recall': recall,
        'fake_f1': f1,
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values('threshold').round(4)
display_and_save_table(threshold_df, 'cleaned_threshold_tradeoff_balanced_linear_svm', index=False)

threshold_plot = threshold_df.melt(
    id_vars=['threshold', 'flagged_rate'],
    value_vars=['fake_precision', 'fake_recall', 'fake_f1'],
    var_name='metric',
    value_name='score'
)
plt.figure(figsize=(9, 5))
sns.lineplot(data=threshold_plot, x='flagged_rate', y='score', hue='metric', marker='o')
plt.title('Threshold Tradeoff: Precision, Recall, and F1')
plt.xlabel('Share of postings flagged for review')
plt.ylabel('Score')
plt.ylim(0, 1)
savefig('cleaned_threshold_tradeoff_precision_recall_f1')
plt.show()

count_plot = threshold_df.melt(
    id_vars=['threshold', 'flagged_rate'],
    value_vars=['true_positives', 'false_positives', 'false_negatives'],
    var_name='outcome',
    value_name='count'
)
plt.figure(figsize=(9, 5))
sns.lineplot(data=count_plot, x='flagged_rate', y='count', hue='outcome', marker='o')
plt.title('Threshold Tradeoff: Review Volume and Mistakes')
plt.xlabel('Share of postings flagged for review')
plt.ylabel('Number of postings')
savefig('cleaned_threshold_tradeoff_counts')
plt.show()

## Final Model Recommendation

The best model depends on what kind of mistake matters more.

- If the goal is to **catch more fake jobs**, use the **balanced Linear SVM**. It has higher fake recall and misses fewer fake postings.
- If the goal is to **avoid falsely accusing real postings**, use the **unweighted Linear SVM**. It has higher fake precision but misses more fake postings.
- For a realistic workflow, the model should be used as a **human-review flag**, not as an automatic removal system.

This makes the final project argument stronger: the model is not just a score-maximization exercise. It is a decision-support system where threshold and metric choices should match the real-world cost of mistakes.

In [ ]:
# Compact recommendation table for the final report
recommendation_table = weighted_results_df[
    (weighted_results_df['dataset'].eq('cleaned')) &
    (weighted_results_df['model'].isin(['Linear SVM balanced', 'Linear SVM unweighted']))
][[
    'model', 'average_precision_mean', 'precision_fake_mean', 'recall_fake_mean', 'f1_fake_mean', 'balanced_accuracy_mean'
]].copy()

recommendation_table['recommended_when'] = recommendation_table['model'].map({
    'Linear SVM balanced': 'Catching more fake postings is the priority',
    'Linear SVM unweighted': 'Avoiding false accusations is the priority',
})

recommendation_table = recommendation_table.sort_values('recall_fake_mean', ascending=False).round(4)
display_and_save_table(recommendation_table, 'final_model_recommendation_tradeoff', index=False)

## Project Conclusion

This project supports the argument that fake job posting detection is best treated as an imbalanced classification problem. Accuracy is not enough because the majority class is so large that a useless majority-class model can still look accurate.

The strongest classic model in this analysis is the class-weighted Linear SVM using TF-IDF text features plus available metadata. The cleaned dataset performs only slightly better than the original dataset, suggesting that most predictive signal is already present in the raw text fields. The practical modeling question is therefore not simply which model has the highest accuracy, but which model creates the best precision-recall tradeoff for identifying rare fake postings.

In a real-world setting, this kind of model should support human review rather than automatically rejecting job posts. False positives could unfairly flag legitimate employers, while false negatives could leave harmful fraudulent postings online.

## Notes for the Final Project

Possible project angle:

Build an interpretable fake job posting detector using classic machine learning, then explain how imbalance changes model evaluation. The cleaned dataset can be framed as the feature-engineered version, while the original dataset is the raw baseline. A good final deliverable would include:

- Class imbalance discussion and why accuracy is misleading.
- EDA findings about missing fields, text length, and high-risk categories.
- Cross-validation table comparing classic models.
- Confusion matrix and classification report for the best model.
- Error analysis of false positives and false negatives.
- A short discussion of ethical limitations: this should support human review, not automatically reject job posts.